Import libraries and functions

In [13]:
import time as time
from poisson_bcann_vi import *
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler


from codebase_0 import CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader, set_seed
from gamma_horseshoe import HorseshoeGammaBCANN, gamma_build_guide_horseshoe_bcann
from gamma_bcann_vi import (
    gamma_build_bcann_guide, GammaBCANN, gamma_fit_glm_mle,
    gamma_bcann_svi_training_loop, GammaCANN, gamma_cann_training_loop,
    gamma_deviance, gamma_weighted_mean, gamma_evaluate_bcann
)
import torch
import pyro

seed = 42
set_seed(seed)
pyro.clear_param_store()

Load data and train test split

In [14]:
device = torch.device("cuda")

torch.set_default_device("cpu")

dataset = pd.read_csv("freMTPL2sev_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimAmount"].to_numpy(), dtype=torch.float32)
weights = torch.ones(len(y)) 
X = torch.tensor(
    dataset.drop(columns=["IDpol", "Exposure","ClaimAmount"]).to_numpy(), 
    dtype=torch.float32
)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, w_learn, w_test = train_test_split(
    X, y, weights, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_learn, y_learn, w_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
w_train, w_val, w_learn, w_test = map(T, (w_train, w_val, w_learn, w_test))

torch.set_default_device(device)
train_loader = DeviceLoader(
        X_train, y_train, w_train,
        batch_size = 2056,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, w_val,
        batch_size= 10000
)
learn_loader = DeviceLoader(
        X_learn, y_learn, w_learn,
        batch_size= 2056
)
test_loader = DeviceLoader(
        X_test, y_test, w_test,
        batch_size= 10000
)

In [ ]:
network_architecture = [20, 15, 10]
gamma_cann = HorseshoeGammaBCANN(
    in_features= 48, 
    neural_net_params= {"hidden_activation_nodes": network_architecture},
).to(device)

glm_mle = gamma_fit_glm_mle(
    X_train, y_train, w_train, y_scale=gamma_cann.y_scale, random_state=seed
)
horseshoe_gamma_bcann_guide = gamma_build_guide_horseshoe_bcann(
    gamma_cann, X_train[:256], 
    y_train[:256],
    w_train[:256],
    glm_mle=glm_mle
)

Start GammaBCANN with horseshoe prior and train its variational parameters

In [ ]:
pyro.clear_param_store()

horseshoe_gamma = HorseshoeGammaBCANN(
    in_features= 48, 
    neural_net_params= {"hidden_activation_nodes": network_architecture},
).to(device)

glm_mle = gamma_fit_glm_mle(X_train, y_train, random_state=seed)
guide = gamma_build_guide_horseshoe_bcann(horseshoe_gamma, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10)

horseshoe_history_full = gamma_bcann_svi_training_loop(
    horseshoe_gamma, guide, 
    train_loader, val_loader, n_epochs=300,
    lr=5e-3, val_every=1, # full 300 epoch runs
    patience=300, random_state=seed
    )

Shape warm-up: alpha held at 0.058 for 20 epochs
Epoch 1 | -ELBO/n: 10.3225 | train deviance: 1.5190 | val deviance: 1.5366 | alpha (median): 0.058 [warm-up]
Epoch 25 | -ELBO/n: 10.1431 | train deviance: 1.5201 | val deviance: 1.5370 | alpha (median): 0.060
Epoch 50 | -ELBO/n: 9.9267 | train deviance: 1.5237 | val deviance: 1.5228 | alpha (median): 0.069
Epoch 75 | -ELBO/n: 9.7744 | train deviance: 1.5248 | val deviance: 1.5331 | alpha (median): 0.079
Epoch 100 | -ELBO/n: 9.6385 | train deviance: 1.5235 | val deviance: 1.5355 | alpha (median): 0.090
Epoch 125 | -ELBO/n: 9.5502 | train deviance: 1.5208 | val deviance: 1.5234 | alpha (median): 0.104
Epoch 150 | -ELBO/n: 9.4505 | train deviance: 1.5303 | val deviance: 1.5243 | alpha (median): 0.119
Epoch 175 | -ELBO/n: 9.3577 | train deviance: 1.5253 | val deviance: 1.5307 | alpha (median): 0.136
Epoch 200 | -ELBO/n: 9.2453 | train deviance: 1.5244 | val deviance: 1.5347 | alpha (median): 0.156
Epoch 225 | -ELBO/n: 9.1603 | train deviance

Best epoch was 191. Training with best epoch

In [5]:
pyro.clear_param_store()
network_architecture = [20, 15, 10]

horseshoe_gamma = HorseshoeGammaBCANN(
    in_features= 48, 
    neural_net_params= {"hidden_activation_nodes": network_architecture},
).to(device)

glm_mle = gamma_fit_glm_mle(X_train, y_train, random_state=seed)
guide = gamma_build_guide_horseshoe_bcann(horseshoe_gamma, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10)

time_0 = time.time()
horseshoe_bcann_history = gamma_bcann_svi_training_loop(
    horseshoe_gamma, guide, 
    train_loader, val_loader, n_epochs=191,
    lr=5e-3, val_every=191, # full 300 epoch runs
    patience=300, random_state=seed
    )
time_1 = time.time()
print(time_1 - time_0)

Shape warm-up: alpha held at 0.058 for 20 epochs
Epoch 1 | -ELBO/n: 10.3225 | train deviance: 1.5190 | val deviance: 1.5366 | alpha (median): 0.058 [warm-up]
148.92432188987732


In [9]:
print(horseshoe_bcann_history.get("best_val_deviance"))
print(horseshoe_bcann_history.get("train_deviance")[-1])

1.5280077457427979
1.5271573066711426


In [ ]:
# write to disk to save runtime
import json

with open("plotting_json/gamma_horseshoe_bcann.json", "w") as f:
    json.dump(horseshoe_history_full, f, indent=2)

Train on full set and get test set performance

In [ ]:
# write to disk for later uses

torch.save({
    "recipe": {
        "cls": "HorseshoeGammaBCANN",
        "kwargs": dict(in_features= 48,
                        neural_net_params={"hidden_activation_nodes": network_architecture}),
    },
    "guide_builder": "gamma_build_guide_horseshoe_bcann",
    "guide_kwargs": dict(rank=10),                     # + tau_init for the horseshoe builders
    "guide_state": {k: v.detach().cpu() for k, v in guide.state_dict().items()},
}, "runs/gamma_horseshoe_bcann_vi.pt")

In [15]:
# GammaBCANN with Gaussian prior on test set
gamma_evaluate_bcann(horseshoe_gamma, guide, test_loader, 200, device = device)

1.8123852014541626

# Sensitivity test

In [7]:
# sensitivity of prediction performance on value of tau
tau_sensi_scores = []
y_scale = gamma_weighted_mean(y_train)

for tau in [1e-3, 1e-2, 1e-1, 1.0, 2.0]:
    pyro.clear_param_store()
    set_seed(seed)
    model = HorseshoeGammaBCANN(
        in_features= 48, 
        neural_net_params= {"hidden_activation_nodes": network_architecture},
        y_scale = y_scale,
        tau_0 = tau
    ).to(device)
    guide = gamma_build_guide_horseshoe_bcann(
        model, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10
    )
    bcann_history = gamma_bcann_svi_training_loop(
        model, guide, train_loader, val_loader,
        n_epochs=300, lr=5e-3, val_every=5, patience=10,
        random_state=seed, device=device, kl_scheme = "uniform")

    tau_sensi_scores.append({"tau": tau, **bcann_history})


Shape warm-up: alpha held at 0.058 for 20 epochs
Epoch 1 | -ELBO/n: 80.0823 | train deviance: 14.1384 | val deviance: 14.0336 | alpha (median): 0.058 [warm-up]
Epoch 25 | -ELBO/n: 10.3098 | train deviance: 1.9683 | val deviance: 2.1202 | alpha (median): 0.060
Epoch 50 | -ELBO/n: 10.0267 | train deviance: 1.7542 | val deviance: 1.8482 | alpha (median): 0.069
Epoch 75 | -ELBO/n: 9.9018 | train deviance: 1.6846 | val deviance: 1.7738 | alpha (median): 0.079
Epoch 100 | -ELBO/n: 9.7953 | train deviance: 1.6361 | val deviance: 1.7193 | alpha (median): 0.090
Epoch 125 | -ELBO/n: 9.6585 | train deviance: 1.6180 | val deviance: 1.6744 | alpha (median): 0.104
Epoch 150 | -ELBO/n: 9.5650 | train deviance: 1.5973 | val deviance: 1.6594 | alpha (median): 0.119
Epoch 175 | -ELBO/n: 9.4415 | train deviance: 1.5880 | val deviance: 1.6437 | alpha (median): 0.136
Epoch 200 | -ELBO/n: 9.3181 | train deviance: 1.5802 | val deviance: 1.6285 | alpha (median): 0.156
Epoch 225 | -ELBO/n: 9.2292 | train devia

In [8]:
# sensitivity of prediction performance on value of slab_scale

slab_scale_sensi_scores = []

for ss in [0.5, 1.0, 2.0, 4.0]:
    pyro.clear_param_store()
    set_seed(seed)
    model = HorseshoeGammaBCANN(
    in_features= 48, 
    neural_net_params= {"hidden_activation_nodes": network_architecture},
    slab_scale = ss).to(device)
    guide = gamma_build_guide_horseshoe_bcann(
        model, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10
    )
    bcann_history = gamma_bcann_svi_training_loop(
        model, guide, train_loader, val_loader,
        n_epochs=300, lr=5e-3, val_every=5, patience=10,
        random_state=seed, device=device, kl_scheme = "uniform")

    slab_scale_sensi_scores.append({"slab_scale": ss, **bcann_history})


Shape warm-up: alpha held at 0.058 for 20 epochs
Epoch 1 | -ELBO/n: 10.2628 | train deviance: 1.5162 | val deviance: 1.5328 | alpha (median): 0.058 [warm-up]
Epoch 25 | -ELBO/n: 10.0393 | train deviance: 1.5167 | val deviance: 1.5313 | alpha (median): 0.060
Epoch 50 | -ELBO/n: 9.8609 | train deviance: 1.5193 | val deviance: 1.5210 | alpha (median): 0.069
Epoch 75 | -ELBO/n: 9.7206 | train deviance: 1.5260 | val deviance: 1.5354 | alpha (median): 0.079
Epoch 100 | -ELBO/n: 9.6140 | train deviance: 1.5219 | val deviance: 1.5267 | alpha (median): 0.090
Early stopping at epoch 105. Best val deviance 1.5175 at epoch 55
Shape warm-up: alpha held at 0.058 for 20 epochs
Epoch 1 | -ELBO/n: 10.2954 | train deviance: 1.5170 | val deviance: 1.5369 | alpha (median): 0.058 [warm-up]
Epoch 25 | -ELBO/n: 10.0493 | train deviance: 1.5171 | val deviance: 1.5290 | alpha (median): 0.060
Epoch 50 | -ELBO/n: 9.8749 | train deviance: 1.5186 | val deviance: 1.5210 | alpha (median): 0.069
Epoch 75 | -ELBO/n: 9

In [23]:
# sensitivity of prediction performance on value of slab_df

slab_df_sensi_scores = []

for df in [2.0, 4.0, 8.0, 25.0]:
    pyro.clear_param_store()
    set_seed(seed)
    model = HorseshoeGammaBCANN(
        in_features= 48, 
        neural_net_params= {
            "hidden_activation_nodes": network_architecture
        },
        slab_df = df
    ).to(device)
    guide = gamma_build_guide_horseshoe_bcann(
        model, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10
    )
    bcann_history = gamma_bcann_svi_training_loop(
        model, guide, train_loader, val_loader,
        n_epochs=300, lr=5e-3, val_every=5, patience=10,
        random_state=seed, device=device, kl_scheme = "uniform")

    slab_scale_sensi_scores.append({"slab_df": df, **bcann_history})


Shape warm-up: alpha held at 0.058 for 20 epochs
Epoch 1 | -ELBO/n: 12.6897 | val deviance: 7.6718 | alpha (median): 0.058 [warm-up]
Epoch 25 | -ELBO/n: 10.0188 | val deviance: 1.6190 | alpha (median): 0.061
Epoch 50 | -ELBO/n: 9.7743 | val deviance: 1.5756 | alpha (median): 0.080
Epoch 75 | -ELBO/n: 9.5649 | val deviance: 1.5599 | alpha (median): 0.104
Epoch 100 | -ELBO/n: 9.3763 | val deviance: 1.5749 | alpha (median): 0.135
Epoch 125 | -ELBO/n: 9.1728 | val deviance: 1.5380 | alpha (median): 0.175
Epoch 150 | -ELBO/n: 9.0142 | val deviance: 1.5305 | alpha (median): 0.227
Early stopping at epoch 165. Best val deviance 1.5159 at epoch 115
Shape warm-up: alpha held at 0.058 for 20 epochs
Epoch 1 | -ELBO/n: 12.6938 | val deviance: 7.6344 | alpha (median): 0.058 [warm-up]
Epoch 25 | -ELBO/n: 10.0215 | val deviance: 1.6218 | alpha (median): 0.061
Epoch 50 | -ELBO/n: 9.7718 | val deviance: 1.5822 | alpha (median): 0.080
Epoch 75 | -ELBO/n: 9.5665 | val deviance: 1.5523 | alpha (median): 0.